In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

rahulm518_m2cai_tool_path = kagglehub.dataset_download('rahulm518/m2cai-tool')
rahulm518_class_list_1_path = kagglehub.dataset_download('rahulm518/class-list-1')
rahulm518_test_vid_path = kagglehub.dataset_download('rahulm518/test-vid')
rahulm518_trained_yolov8_path = kagglehub.dataset_download('rahulm518/trained-yolov8')
rahulm518_best_seg_path = kagglehub.dataset_download('rahulm518/best-seg')
rahulm518_saved_yaml_path = kagglehub.dataset_download('rahulm518/saved-yaml')
rahulm518_test_seg_path = kagglehub.dataset_download('rahulm518/test-seg')
rahulm518_yolov5py_path = kagglehub.dataset_download('rahulm518/yolov5py')
rahulm518_helpme_path = kagglehub.dataset_download('rahulm518/helpme')

print('Data source import complete.')


100%|██████████| 136M/136M [00:01<00:00, 79.3MB/s]

Extracting files...


100%|██████████| 218/218 [00:00<00:00, 370kB/s]

Extracting files...


100%|██████████| 193M/193M [00:01<00:00, 164MB/s]


...

Extracting files...


100%|██████████| 10.9k/10.9k [00:00<00:00, 13.0MB/s]

Extracting files...


100%|██████████| 10.9k/10.9k [00:00<00:00, 15.1MB/s]

Extracting files...
Data source import complete.


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.3 MB/s eta 0:00:00


In [11]:
import os
import shutil

# 1. Create the /kaggle/input/ directory (if it doesn't exist)
os.makedirs("/kaggle/working/", exist_ok=True)

# 2. Pick your dataset paths from kagglehub
# For example:
dataset_src = rahulm518_m2cai_tool_path  # This is from kagglehub

# 3. Copy to /kaggle/input/ with a proper folder name
shutil.copytree(dataset_src, "/kaggle/working/m2cai-tool", dirs_exist_ok=True)
shutil.copytree(rahulm518_class_list_1_path, "/kaggle/working/class-list-1", dirs_exist_ok=True)
shutil.copytree(rahulm518_test_vid_path, "/kaggle/working/test-vid", dirs_exist_ok=True)
shutil.copytree(rahulm518_trained_yolov8_path, "/kaggle/working/trained-yolov8", dirs_exist_ok=True)
shutil.copytree(rahulm518_best_seg_path, "/kaggle/working/best-seg", dirs_exist_ok=True)
shutil.copytree(rahulm518_saved_yaml_path, "/kaggle/working/saved-yaml", dirs_exist_ok=True)
shutil.copytree(rahulm518_test_seg_path, "/kaggle/working/test-seg", dirs_exist_ok=True)
shutil.copytree(rahulm518_yolov5py_path, "/kaggle/working/yolov5py", dirs_exist_ok=True)
shutil.copytree(rahulm518_helpme_path, "/kaggle/working/helpme", dirs_exist_ok=True)

# 4. Now it's accessible at /kaggle/input/m2cai-tool
print("Copied to /kaggle/working/m2cai-tool")

# Optionally, list files to verify
!ls /kaggle/input/m2cai-tool


Copied to /kaggle/working/m2cai-tool
ls: cannot access '/kaggle/input/m2cai-tool': No such file or directory


In [12]:
import os
DATASET_PATH = "/kaggle/working/m2cai-tool/m2cai16-tool-locations"

if(1):
  ANNOTATIONS_DIR = "/kaggle/working/m2cai-tool/m2cai16-tool-locations/Annotations"
  IMAGES_DIR = "/kaggle/working/m2cai-tool/m2cai16-tool-locations/JPEGImages"
  YOLO_DIR = "/kaggle/working/m2cai-yolo"
  CLASS_LIST_FILE = "/kaggle/working/class-list-1/class_list.txt"

# Create folders
os.makedirs(f"{YOLO_DIR}/images/train", exist_ok=True)
os.makedirs(f"{YOLO_DIR}/images/val", exist_ok=True)
os.makedirs(f"{YOLO_DIR}/images/test", exist_ok=True)
os.makedirs(f"{YOLO_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{YOLO_DIR}/labels/val", exist_ok=True)
os.makedirs(f"{YOLO_DIR}/labels/test", exist_ok=True)

In [5]:
print("✅ Example check:", os.path.exists("/kaggle/input/m2cai-tool/m2cai16-tool-locations/JPEGImages/v03_114050.jpg"))

✅ Example check: False


In [6]:
import os
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
import shutil

# === Load class list ===
with open(CLASS_LIST_FILE, "r") as f:
    class_names = [line.strip() for line in f.readlines()]
class_to_id = {name: idx for idx, name in enumerate(class_names)}
print("✅ Loaded classes:", class_to_id)

# === Setup output folders ===
for split in ["train", "val", "test"]:
    os.makedirs(f"{YOLO_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_DIR}/labels/{split}", exist_ok=True)

# === Load all XML files ===
xml_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith(".xml")]
print(f"📁 Found {len(xml_files)} annotation files.")

# === Split the dataset ===
train_files, temp_files = train_test_split(xml_files, test_size=0.30, random_state=42)
val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

# === VOC to YOLO Converter ===
def convert_annotation(xml_path, output_path, img_filename):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        img_w = int(root.find("size/width").text)
        img_h = int(root.find("size/height").text)

        label_lines = []
        for obj in root.findall("object"):
            cls = obj.find("name").text.strip().lower()
            if cls not in class_to_id:
                print(f"⚠️ Unknown class '{cls}' in {img_filename}, skipping.")
                continue

            cls_id = class_to_id[cls]

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            # YOLO format
            x_center = ((xmin + xmax) / 2) / img_w
            y_center = ((ymin + ymax) / 2) / img_h
            width = (xmax - xmin) / img_w
            height = (ymax - ymin) / img_h

            label_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        if label_lines:
            with open(output_path, "w") as f:
                f.write("\n".join(label_lines))
        else:
            print(f"⚠️ No objects found in {img_filename}. Skipping label file.")
    except Exception as e:
        print(f"❌ Error in {img_filename}: {e}")

# === Copy and Convert ===
def process_set(file_list, set_type):
    print(f"🔄 Processing {set_type} set with {len(file_list)} samples...")
    for xml_file in file_list:
        #file_id = xml_file.replace(".xml", "")
        file_id = os.path.splitext(os.path.basename(xml_file))[0]
        img_file = f"{file_id}.jpg"
        src_img = os.path.join(IMAGES_DIR, img_file)
        #print(f"🧪 DEBUG img_file = {img_file}")
        dst_img = os.path.join(YOLO_DIR, f"images/{set_type}", img_file)
        label_path = os.path.join(YOLO_DIR, f"labels/{set_type}", f"{file_id}.txt")

        if not os.path.exists(src_img):
            print(f"⚠️ Missing image: {src_img}")
            continue

        shutil.copy2(src_img, dst_img)
        convert_annotation(os.path.join(ANNOTATIONS_DIR, xml_file), label_path, file_id)

# === Run ===
process_set(train_files, "train")
process_set(val_files, "val")
process_set(test_files, "test")

print("✅ Conversion complete!")


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/class-list-1/class_list.txt'

In [ ]:
import yaml

data_yaml = {
    'train': '/kaggle/working/m2cai-yolo/images/train',
    'val': '/kaggle/working/m2cai-yolo/images/val',
    'test': '/kaggle/working/m2cai-yolo/images/test',
    'nc': 7,
    'names': ['Grasper', 'Bipolar', 'Hook', 'Scissors', 'Clipper', 'Irrigator', 'SpecimenBag']
}

with open('/kaggle/working/m2cai-yolo/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("✅ data.yaml file created in /kaggle/working/m2cai-yolo/")


✅ data.yaml file created in /kaggle/working/m2cai-yolo/


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Or yolov8s.pt, yolov8m.pt, etc.
model.train(data="/kaggle/working/m2cai-yolo/data.yaml", epochs=50, imgsz=640, batch=16, name="m2cai_yolo")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 6.25M/6.25M [00:00<00:00, 162MB/s]


Ultralytics 8.3.135 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/m2cai-yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=m2cai_yolo, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=Tr

100%|██████████| 755k/755k [00:00<00:00, 35.4MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 14/14 [00:02<00:00,  5.31it/s]


                   all        422        581      0.961      0.931      0.968      0.636
               Grasper        164        204      0.936      0.854      0.932      0.573
               Bipolar         65         65      0.974      0.938      0.961      0.611
                  Hook         44         44          1       0.99      0.995      0.789
              Scissors         60         60      0.964          1       0.99      0.649
               Clipper         66         66      0.962      0.924      0.971      0.659
             Irrigator         64         64      0.966       0.89      0.962      0.521
           SpecimenBag         78         78      0.923      0.923      0.967      0.652


/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs/detect/m2cai_yolo


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bf4a33b6ce0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  

In [ ]:
metrics = model.val(split='test')  # Use test split
print(metrics)

Ultralytics 8.3.135 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 649.2±307.4 MB/s, size: 28.9 KB)


val: Scanning /kaggle/working/m2cai-yolo/labels/test... 422 images, 0 backgrounds, 0 corrupt: 100%|██████████| 422/422 [00:00<00:00, 1161.21it/s]

val: New cache created: /kaggle/working/m2cai-yolo/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 27/27 [00:03<00:00,  8.68it/s]


                   all        422        613      0.963      0.933      0.968       0.62
               Grasper        197        240      0.909      0.837      0.908      0.561
               Bipolar         71         71      0.984      0.972      0.987       0.61
                  Hook         51         51      0.992      0.961      0.982      0.712
              Scissors         52         52       0.93      0.942      0.956      0.586
               Clipper         57         57      0.981      0.909      0.987      0.681
             Irrigator         68         68      0.985      0.966      0.989      0.584
           SpecimenBag         74         74      0.959      0.944      0.967      0.606


/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


Speed: 0.5ms preprocess, 1.8ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs/detect/m2cai_yolo2
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bf4a0c9d420>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.0380

In [ ]:
results = model.predict(source="/kaggle/working/m2cai-yolo/images/test", save=True, conf=0.25)


image 1/422 /kaggle/working/m2cai-yolo/images/test/v01_004225.jpg: 384x640 1 Grasper, 36.1ms
image 2/422 /kaggle/working/m2cai-yolo/images/test/v01_007125.jpg: 384x640 2 Graspers, 1 SpecimenBag, 6.6ms
image 3/422 /kaggle/working/m2cai-yolo/images/test/v01_020150.jpg: 384x640 2 Graspers, 6.9ms
image 4/422 /kaggle/working/m2cai-yolo/images/test/v01_027600.jpg: 384x640 2 Graspers, 1 Scissors, 6.5ms
image 5/422 /kaggle/working/m2cai-yolo/images/test/v01_027625.jpg: 384x640 2 Graspers, 1 Scissors, 6.7ms
image 6/422 /kaggle/working/m2cai-yolo/images/test/v01_027875.jpg: 384x640 2 Graspers, 1 Scissors, 6.6ms
image 7/422 /kaggle/working/m2cai-yolo/images/test/v01_028050.jpg: 384x640 2 Graspers, 1 Scissors, 6.8ms
image 8/422 /kaggle/working/m2cai-yolo/images/test/v01_028150.jpg: 384x640 2 Graspers, 1 Scissors, 6.9ms
image 9/422 /kaggle/working/m2cai-yolo/images/test/v01_028175.jpg: 384x640 2 Graspers, 1 Scissors, 7.0ms
image 10/422 /kaggle/working/m2cai-yolo/images/test/v01_028250.jpg: 384x640

In [ ]:
print(results[0].save_dir)


runs/detect/m2cai_yolo3


In [ ]:
import os
from IPython.display import Image, display

pred_dir = "/kaggle/working/runs/detect/predict/m2cai_yolo33"
display(Image(filename="/kaggle/working/runs/detect/m2cai_yolo33/v01_004225.jpg"))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/detect/m2cai_yolo33/v01_004225.jpg'

In [ ]:
import os
from IPython.display import Image, display

pred_dir = "/kaggle/working/runs/detect/m2cai_yolo3"

# Loop through and display all images in the predictions folder
for img_file in sorted(os.listdir(pred_dir)):
    if img_file.lower().endswith((".jpg", ".png", ".jpeg")):
        img_path = os.path.join(pred_dir, img_file)
        display(Image(filename=img_path))


In [ ]:
from ultralytics import YOLO

# Load your trained model - saved after previous step to prevent any loss of model (or redo the entire process, with this cell too)
model = YOLO("/kaggle/input/trained-yolov8/best.pt")  # Update if path is different

# Run inference on your test video
results = model.predict(source="/kaggle/input/test-vid/shortened3min_vid.mp4", save=True, conf=0.25)

print("✅ Inference done! Output saved in:", results[0].save_dir)

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from ultralytics import YOLO
import torch
# Load your segmentation model - https://www.kaggle.com/code/anmolys/cholecseg8k-yolov8/output
model = YOLO("/kaggle/input/best-seg/best _seg.pt")  # Update if path is different

# Run inference on your test video
results = model.predict(
    source="/kaggle/input/test-vid/shortened3min_vid.mp4",
    save=True,
    conf=0.25,
    device="cpu"
)

print("✅ Inference done! Output saved in:", results[0].save_dir)
torch.cuda.empty_cache()


In [ ]:
from moviepy.editor import VideoFileClip

# Load .avi and convert to .mp4
input_path = "/kaggle/working/runs/detect/predict/shortened3min_vid.avi"
output_path = "/kaggle/working/predicted_output.mp4"

clip = VideoFileClip(input_path)
clip.write_videofile(output_path, codec='libx264')

print("✅ Video converted to MP4:", output_path)


In [ ]:
# Install Ultralytics if not already
!pip install ultralytics

# Import YOLO
from ultralytics import YOLO

# Load your model
model = YOLO('/kaggle/input/trained-yolov8/best.pt')  # e.g., 'runs/segment/train/weights/best.pt' after training

# Run validation
# This will calculate precision, recall, F1 score, IoU for each class
metrics = model.val(
    data='/kaggle/input/saved-yaml/data.yaml',    # Your dataset config
    split='val',                 # Use validation set
    save_json=True,              # Save COCO-style metrics
    iou=0.5                      # IoU threshold
)

# Now you can access detailed metrics
print("Overall Metrics:")
print(f"Precision: {metrics.box.p.mean():.4f}")
print(f"Recall: {metrics.box.r.mean():.4f}")
print(f"mAP50: {metrics.box.map50.mean():.4f}")
print(f"mAP50-95: {metrics.box.map.mean():.4f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load your results.csv
df = pd.read_csv('/kaggle/input/test-seg/results.csv')

# Clean up column names
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('"', '')  # just in case quotes are there

print(df.columns.tolist())  # see cleaned column names

# Now extract final epoch
final_epoch = df.iloc[-1]

print("\n=== Final Model Performance ===")
print(f"Precision (B): {final_epoch['metrics/precision(B)']:.4f}")
print(f"Recall    (B): {final_epoch['metrics/recall(B)']:.4f}")
print(f"mAP50     (B): {final_epoch['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95  (B): {final_epoch['metrics/mAP50-95(B)']:.4f}")

print(f"\nPrecision (M): {final_epoch['metrics/precision(M)']:.4f}")
print(f"Recall    (M): {final_epoch['metrics/recall(M)']:.4f}")
print(f"mAP50     (M): {final_epoch['metrics/mAP50(M)']:.4f}")
print(f"mAP50-95  (M): {final_epoch['metrics/mAP50-95(M)']:.4f}")


# How do I get an YOLOv5 for FPGA


In [13]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17485, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 17485 (delta 81), reused 31 (delta 31), pack-reused 17374 (from 3)
Receiving objects: 100% (17485/17485), 16.39 MiB | 17.26 MiB/s, done.
Resolving deltas: 100% (11986/11986), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/

In [14]:
YOLO_PATHv5 = "/kaggle/working/m2cai-yolo-v5"
os.makedirs(YOLO_PATHv5, exist_ok=True)

In [15]:
os.makedirs(f"{YOLO_PATHv5}/images/train", exist_ok=True)
os.makedirs(f"{YOLO_PATHv5}/images/val", exist_ok=True)
os.makedirs(f"{YOLO_PATHv5}/images/test", exist_ok=True)
os.makedirs(f"{YOLO_PATHv5}/labels/train", exist_ok=True)
os.makedirs(f"{YOLO_PATHv5}/labels/val", exist_ok=True)
os.makedirs(f"{YOLO_PATHv5}/labels/test", exist_ok=True)

In [16]:
import os
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
import shutil

# === Load class list ===
with open(CLASS_LIST_FILE, "r") as f:
    class_names = [line.strip() for line in f.readlines()]
class_to_id = {name: idx for idx, name in enumerate(class_names)}
print("✅ Loaded classes:", class_to_id)

# === Setup output folders ===
for split in ["train", "val", "test"]:
    os.makedirs(f"{YOLO_PATHv5}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_PATHv5}/labels/{split}", exist_ok=True)

# === Load all XML files ===
xml_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith(".xml")]
print(f"📁 Found {len(xml_files)} annotation files.")

# === Split the dataset ===
train_files, temp_files = train_test_split(xml_files, test_size=0.30, random_state=42)
val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

# === VOC to YOLO Converter ===
def convert_annotation(xml_path, output_path, img_filename):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        img_w = int(root.find("size/width").text)
        img_h = int(root.find("size/height").text)

        label_lines = []
        for obj in root.findall("object"):
            cls = obj.find("name").text.strip().lower()
            if cls not in class_to_id:
                print(f"⚠️ Unknown class '{cls}' in {img_filename}, skipping.")
                continue

            cls_id = class_to_id[cls]

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            # YOLO format
            x_center = ((xmin + xmax) / 2) / img_w
            y_center = ((ymin + ymax) / 2) / img_h
            width = (xmax - xmin) / img_w
            height = (ymax - ymin) / img_h

            label_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        if label_lines:
            with open(output_path, "w") as f:
                f.write("\n".join(label_lines))
        else:
            print(f"⚠️ No objects found in {img_filename}. Skipping label file.")
    except Exception as e:
        print(f"❌ Error in {img_filename}: {e}")

# === Copy and Convert ===
def process_set(file_list, set_type):
    print(f"🔄 Processing {set_type} set with {len(file_list)} samples...")
    for xml_file in file_list:
        #file_id = xml_file.replace(".xml", "")
        file_id = os.path.splitext(os.path.basename(xml_file))[0]
        img_file = f"{file_id}.jpg"
        src_img = os.path.join(IMAGES_DIR, img_file)
        #print(f"🧪 DEBUG img_file = {img_file}")
        dst_img = os.path.join(YOLO_PATHv5, f"images/{set_type}", img_file)
        label_path = os.path.join(YOLO_PATHv5, f"labels/{set_type}", f"{file_id}.txt")

        if not os.path.exists(src_img):
            print(f"⚠️ Missing image: {src_img}")
            continue

        shutil.copy2(src_img, dst_img)
        convert_annotation(os.path.join(ANNOTATIONS_DIR, xml_file), label_path, file_id)

# === Run ===
process_set(train_files, "train")
process_set(val_files, "val")
process_set(test_files, "test")

print("✅ Conversion complete!")


✅ Loaded classes: {'grasper': 0, 'bipolar': 1, 'hook': 2, 'scissors': 3, 'clipper': 4, 'irrigator': 5, 'specimenbag': 6}
📁 Found 2811 annotation files.
🔄 Processing train set with 1967 samples...
🔄 Processing val set with 422 samples...
🔄 Processing test set with 422 samples...
✅ Conversion complete!


In [17]:
import yaml

data_yaml = {
    'train': '/kaggle/working/m2cai-yolo-v5/images/train',
    'val': '/kaggle/working/m2cai-yolo-v5/images/val',
    'test': '/kaggle/working/m2cai-yolo-v5/images/test',
    'nc': 7,
    'names': ['Grasper', 'Bipolar', 'Hook', 'Scissors', 'Clipper', 'Irrigator', 'SpecimenBag']
}

with open('/kaggle/working/m2cai-yolo-v5/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("✅ data.yaml file created in /kaggle/working/m2cai-yolo-v5/")


✅ data.yaml file created in /kaggle/working/m2cai-yolo-v5/


In [20]:
import os

# Set this before importing wandb
os.environ['WANDB__REQUIRE_LEGACY_SERVICE'] = 'TRUE'

import wandb

# Now you can login or initialize
wandb.login()


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rahul-milano-blr (rahul-milano-blr-international-institute-of-information-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [21]:
! python train.py --img 640 --batch 16 --epochs 50 --data /kaggle/working/m2cai-yolo-v5/data.yaml --weights yolov5n.pt

Streaming output truncated to the last 5000 lines.
  with torch.cuda.amp.autocast(amp):
      30/49      2.17G    0.03328    0.01834   0.004921         49        640:  35% 43/123 [00:15<00:28,  2.77it/s]/content/yolov5/train.py:413: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      30/49      2.17G    0.03323    0.01834   0.004915         50        640:  36% 44/123 [00:15<00:25,  3.14it/s]/content/yolov5/train.py:413: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      30/49      2.17G    0.03322    0.01837   0.004875         53        640:  37% 45/123 [00:15<00:29,  2.68it/s]/content/yolov5/train.py:413: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp

In [23]:
!python detect.py \
  --weights /content/yolov5/runs/train/exp2/weights/best.pt\
  --img 640 \
  --conf 0.25 \
  --source //content/v08_016225.jpg \
  --save-txt \
  --save-conf \
  --name yolov5n_results


detect: weights=['/content/yolov5/runs/train/exp2/weights/best.pt'], source=//content/v08_016225.jpg, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=True, save_format=0, save_csv=False, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=yolov5n_results, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-418-ga493afe1 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 1768636 parameters, 0 gradients, 4.2 GFLOPs
image 1/1 /content/v08_016225.jpg: 384x640 1 Grasper, 1 Bipolar, 31.4ms
Speed: 0.5ms pre-process, 31.4ms inference, 138.2ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/yolov5n_results
1 labels saved to runs/detect/yolov5n_results/labels


In [24]:
!python val.py \
  --weights /content/yolov5/runs/train/exp2/weights/best.pt \
  --data /kaggle/working/m2cai-yolo-v5/data.yaml \
  --img 640 \
  --task test


val: data=/kaggle/working/m2cai-yolo-v5/data.yaml, weights=['/content/yolov5/runs/train/exp2/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-418-ga493afe1 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 1768636 parameters, 0 gradients, 4.2 GFLOPs
test: Scanning /kaggle/working/m2cai-yolo-v5/labels/test... 422 images, 0 backgrounds, 0 corrupt: 100% 422/422 [00:00<00:00, 1971.12it/s]
test: New cache created: /kaggle/working/m2cai-yolo-v5/labels/test.cache
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 14/14 [00:05<00:00,  2.40it/s]
                   all        422        578      0.965      0.936      0.966      0.5